# Data Quality y EDA orientado a decisiones
Este notebook explica los resultados; la lógica ejecutable vive en `src/` y es la misma que usa producción.

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
from src.config import FIGURES_DIR, OUTPUT_DIR
from src.data import load_adult
from src.quality import run_quality_gates
from src.eda import build_eda_decisions
df = load_adult()
df.shape

(48842, 15)

## G. Data Quality Gates
Las reglas se ejecutan antes del entrenamiento. Un `FAIL` guarda el reporte y bloquea el proceso.

In [2]:
quality_report = run_quality_gates(df, OUTPUT_DIR / 'data_quality_gates.csv')
quality_report

,regla,descripcion,estado,detalle
0,schema,Están presentes las 15 columnas esperadas,PASS,completo
1,minimum_rows,Hay suficientes observaciones,PASS,"48,842 filas; mínimo=1,000"
2,duplicate_ratio,Duplicados exactos no superan 1%,PASS,0.106%; máximo=1.0%
3,target_complete,El target no contiene nulos,PASS,0 nulos
4,target_domain,El target contiene exactamente las dos clases,PASS,"observadas=['<=50K', '>50K']"
5,feature_missingness,Ninguna feature supera 10% de nulos,PASS,máximo=5.75% (occupation); límite=10%
6,numeric_ranges,Variables numéricas respetan rangos plausibles,PASS,rangos válidos


## H. EDA
La tabla siguiente es el entregable central: cada resultado termina en una decisión verificable, no solamente en una figura.

In [3]:
decisions = build_eda_decisions(df, FIGURES_DIR)
decisions.to_csv(OUTPUT_DIR / 'eda_decisiones.csv', index=False)
decisions

,analisis,resultado,decision
0,Balance de clases,clase minoritaria=23.93%,"Partición estratificada, class_weight='balance..."
1,Valores faltantes,máximo=5.75% en occupation,No eliminar filas; imputar dentro del pipeline...
2,Redundancia education,máximo de códigos por categoría=1,Conservar ambas representaciones para el model...
3,Ceros en variables de capital,gain=91.74%; loss=95.33%,No tratar ceros como faltantes; comparar model...
4,Horas de trabajo,la tasa >50K cambia entre bandas de horas,Conservar hours-per-week y comparar modelos li...


## Conclusión
El desbalance determina el split, las métricas y los pesos; los faltantes se imputan dentro del pipeline; los ceros de capital se conservan; y las relaciones no lineales justifican el baseline de árboles.